# CJK Research Conclusion — Comparing Clip Duration Across the cjk-research Notebook Family

This notebook aggregates results from `cjk-research-2sec.ipynb`, `cjk-research-5sec.ipynb`, `cjk-research-7sec.ipynb`, and `cjk-research-11sec.ipynb` — each identical except for `CLIP_DURATION_S` (2, 5, 7, 11 seconds; all at `SAMPLE_RATE = 44100`) — and adds a fresh, matching feature-statistics sweep so every duration can be compared side by side in one place.

**What's compared:**
- Classifier accuracy / F1 (SVM, Random Forest, soft-vote ensemble) per duration — the real numbers already computed and printed in each notebook's "Classifier Sanity Checks" section, not re-run here (5-fold CV is expensive and was already done once per notebook).
- Mean value of each of the 8 hand-crafted audio features per duration — recomputed here using the exact same sampling (20 files/class, `random_state=42`) as every source notebook, since the full feature table isn't persisted outside a notebook's own kernel — only a `.head()` preview is.
- Dataset-level facts that don't vary by duration (file counts, native sample-rate distribution) — cited once from `cjk-research-2sec.ipynb`'s "Dataset At A Glance," not recomputed.

## Setup

Same imports, config, and helper functions as the `cjk-research-*sec.ipynb` family, condensed into one cell since this notebook's job is aggregation, not exploration.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import librosa
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display

SAMPLE_RATE = 44100
RANDOM_SEED = 42
DURATIONS_TO_COMPARE = [2.0, 5.0, 7.0, 11.0]

REPO_ROOT = Path.cwd()
DATA_CANDIDATES = [
    REPO_ROOT / "data" / "raw" / "CatSound_originals",
    REPO_ROOT.parent / "data" / "raw" / "CatSound_originals",
    Path("../data/raw/CatSound_originals"),
]
DATA_DIR = next((c for c in DATA_CANDIDATES if c.exists()), DATA_CANDIDATES[-1])
AUDIO_EXTENSIONS = {".mp3", ".wav", ".flac", ".m4a", ".ogg"}


def is_audio_file(path: Path) -> bool:
    return path.suffix.lower() in AUDIO_EXTENSIONS


def fit_to_duration(audio: np.ndarray, sample_rate: int, duration_s: float) -> np.ndarray:
    target_length = int(round(duration_s * sample_rate))
    current_length = len(audio)
    if current_length < target_length:
        total_pad = target_length - current_length
        pad_before = total_pad // 2
        pad_after = total_pad - pad_before
        return np.pad(audio, (pad_before, pad_after), mode="constant")
    if current_length > target_length:
        total_trim = current_length - target_length
        trim_before = total_trim // 2
        return audio[trim_before:trim_before + target_length]
    return audio


def show_plotly(fig):
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))

## Dataset

Rebuilds the same file inventory (`df`) as the `cjk-research-*sec.ipynb` notebooks — needed so the feature sweep below samples the exact same files, in the exact same order, per class.

In [2]:
classes = sorted([item.name for item in DATA_DIR.iterdir() if item.is_dir()])
audio_files = []
for label in classes:
    for path in sorted((DATA_DIR / label).iterdir()):
        if path.is_file() and is_audio_file(path):
            audio_files.append({"label": label, "filename": path.name, "path": path})

df = pd.DataFrame(audio_files)
print(f"{len(classes)} classes, {len(df)} audio files")

10 classes, 2953 audio files


## Feature Statistics Sweep Across Durations

For each `CLIP_DURATION_S` in {2, 5, 7, 11}, samples the same 20 files per class (`random_state=42`, matching every `cjk-research-*sec.ipynb`) and computes the mean of each of the 8 hand-crafted features. This is the one piece of analysis this notebook actually performs itself — everything else below is aggregated from already-executed results.

In [3]:
feature_columns = ["rms", "zcr", "spectral_centroid", "spectral_bandwidth", "spectral_rolloff", "mfcc_1", "mfcc_2"]

sweep_rows = []
for duration in DURATIONS_TO_COMPARE:
    feature_samples = []
    for label, frame in df.groupby("label"):
        sample = frame.sample(n=min(20, len(frame)), random_state=RANDOM_SEED)
        feature_samples.append(sample[["label", "filename", "path"]])
    feature_sample = pd.concat(feature_samples, ignore_index=True)

    rows = []
    for label, filename, path in feature_sample.itertuples(index=False, name=None):
        y, sr = librosa.load(str(path), sr=SAMPLE_RATE, mono=True)
        y = fit_to_duration(y, sr, duration)
        rows.append(
            {
                "label": label,
                "rms": float(librosa.feature.rms(y=y).mean()),
                "zcr": float(librosa.feature.zero_crossing_rate(y).mean()),
                "spectral_centroid": float(librosa.feature.spectral_centroid(y=y, sr=sr).mean()),
                "spectral_bandwidth": float(librosa.feature.spectral_bandwidth(y=y, sr=sr).mean()),
                "spectral_rolloff": float(librosa.feature.spectral_rolloff(y=y, sr=sr).mean()),
                "mfcc_1": float(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=1).mean()),
                "mfcc_2": float(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=2)[1].mean()),
            }
        )
    duration_feature_df = pd.DataFrame(rows)
    summary = duration_feature_df[feature_columns].mean()
    summary["clip_duration_s"] = duration
    sweep_rows.append(summary)
    print(f"CLIP_DURATION_S={duration}: done")

feature_sweep = pd.DataFrame(sweep_rows).set_index("clip_duration_s")
display(feature_sweep.round(4))

CLIP_DURATION_S=2.0: done


CLIP_DURATION_S=5.0: done


CLIP_DURATION_S=7.0: done


CLIP_DURATION_S=11.0: done


,rms,zcr,spectral_centroid,spectral_bandwidth,spectral_rolloff,mfcc_1,mfcc_2
clip_duration_s,,,,,,,
2.0,0.0433,0.0742,2555.8503,2621.4471,5064.3125,-403.6862,140.7765
5.0,0.0304,0.0551,1953.6469,2047.8423,3925.2469,-479.4036,104.4379
7.0,0.0228,0.0423,1502.9670,1579.2132,3025.6804,-535.5220,79.8745
11.0,0.0145,0.0270,962.4032,1011.1320,1938.1472,-602.2570,51.0932


### Feature Means by Clip Duration

In [4]:
fig = make_subplots(rows=2, cols=4, subplot_titles=feature_columns)
for i, feat in enumerate(feature_columns):
    r, c = divmod(i, 4)
    fig.add_trace(
        go.Scatter(x=feature_sweep.index, y=feature_sweep[feat], mode="lines+markers", name=feat),
        row=r + 1, col=c + 1,
    )
fig.update_layout(height=480, showlegend=False, title="Mean feature value by CLIP_DURATION_S")
fig.update_xaxes(title_text="seconds")
show_plotly(fig)

## Classifier Accuracy Across Durations

Pulled directly from the "Classifier Sanity Checks" output already executed in each `cjk-research-*sec.ipynb` — not recomputed here, since the 5-fold cross-validation was already run per notebook and is expensive to repeat.

In [5]:
classifier_results = pd.DataFrame([
    {"clip_duration_s": 2, "model": "svm_rbf", "accuracy": 0.455, "accuracy_std": 0.081, "f1_macro": 0.440},
    {"clip_duration_s": 2, "model": "random_forest", "accuracy": 0.430, "accuracy_std": 0.073, "f1_macro": 0.409},
    {"clip_duration_s": 2, "model": "soft_vote", "accuracy": 0.435, "accuracy_std": 0.073, "f1_macro": 0.413},
    {"clip_duration_s": 5, "model": "svm_rbf", "accuracy": 0.460, "accuracy_std": 0.086, "f1_macro": 0.445},
    {"clip_duration_s": 5, "model": "random_forest", "accuracy": 0.425, "accuracy_std": 0.055, "f1_macro": 0.411},
    {"clip_duration_s": 5, "model": "soft_vote", "accuracy": 0.450, "accuracy_std": 0.088, "f1_macro": 0.427},
    {"clip_duration_s": 7, "model": "svm_rbf", "accuracy": 0.440, "accuracy_std": 0.107, "f1_macro": 0.427},
    {"clip_duration_s": 7, "model": "random_forest", "accuracy": 0.420, "accuracy_std": 0.053, "f1_macro": 0.414},
    {"clip_duration_s": 7, "model": "soft_vote", "accuracy": 0.435, "accuracy_std": 0.089, "f1_macro": 0.407},
    {"clip_duration_s": 11, "model": "svm_rbf", "accuracy": 0.440, "accuracy_std": 0.093, "f1_macro": 0.428},
    {"clip_duration_s": 11, "model": "random_forest", "accuracy": 0.430, "accuracy_std": 0.037, "f1_macro": 0.416},
    {"clip_duration_s": 11, "model": "soft_vote", "accuracy": 0.415, "accuracy_std": 0.072, "f1_macro": 0.388},
])
display(classifier_results.pivot(index="clip_duration_s", columns="model", values="accuracy").round(3))

model,random_forest,soft_vote,svm_rbf
clip_duration_s,,,
2,0.430,0.435,0.455
5,0.425,0.450,0.460
7,0.420,0.435,0.440
11,0.430,0.415,0.440


### Accuracy and F1 Charts

In [6]:
acc_fig = px.line(
    classifier_results,
    x="clip_duration_s",
    y="accuracy",
    color="model",
    markers=True,
    error_y="accuracy_std",
    title="Classifier accuracy vs. clip duration (5-fold CV, error bars = fold std)",
)
acc_fig.update_layout(xaxis_title="CLIP_DURATION_S (seconds)", yaxis_title="Accuracy")
show_plotly(acc_fig)

f1_fig = px.bar(
    classifier_results,
    x="clip_duration_s",
    y="f1_macro",
    color="model",
    barmode="group",
    title="Macro F1 vs. clip duration",
)
f1_fig.update_layout(xaxis_title="CLIP_DURATION_S (seconds)", yaxis_title="F1 (macro)")
show_plotly(f1_fig)

**Reading these results:** accuracy stays in a narrow band (~0.42-0.46) across all four durations — there's no clear monotonic trend where longer or shorter clips help. Differences between durations are comparable in size to each model's own cross-validation std (0.04-0.11), meaning **duration choice alone doesn't meaningfully move classifier performance** for this hand-crafted-feature pipeline on this dataset. That matters for a future API: a shorter clip (2s) captures audio and predicts faster with no measurable accuracy cost, favoring `CLIP_DURATION_S = 2.0` over longer windows on a pure latency basis — see the earlier discussion in `cjk-research-2sec.ipynb` on why sample rate and clip duration matter for a production API.

## Comparison to the Original Paper

The historical baseline is Pandeya et al. (2018), trained on `CatSound_DataSet_V2` with deep CNN/CDBN features pooled two ways (GAP, FDAP) and tested at increasing levels of data augmentation:

| Source | Features | Data | Accuracy |
|---|---|---|---|
| Paper, Table 3 — no augmentation (Original, GAP pooling) | Deep CNN/CDBN pooled features | Original dataset, full size | 70.7% |
| Paper, Table 3 — 3x augmentation (FDAP pooling) | Same | 3x-augmented dataset, full size | 91.1% |
| Paper, Table 2 — best single classifier (SVM, 3x-aug, FDAP) | Same | 3x-augmented dataset, full size | 87.4–90.9% |
| Paper, Table 2 — best ensemble (3x-aug, FDAP) | Same | 3x-augmented dataset, full size | 90.8–91.1% |
| `.keep/EDA/cjk-audio-conv-7-EDA.ipynb` (archived, this project) | PANNs Cnn14 deep embeddings (2048-dim) | Full 2953-file originals-only set | SVM 81.8%, RF 83.7%, Ensemble 83.3% |
| **This notebook — hand-crafted features, across `CLIP_DURATION_S` 2–11s** | 8 hand-crafted scalar features | 20-per-class stratified sample (~200 files) | 41.5–46.0% accuracy |

In [7]:
paper_benchmarks = pd.DataFrame([
    {"source": "This notebook — hand-crafted features (best of 12)", "accuracy": classifier_results["accuracy"].max()},
    {"source": "Paper — no augmentation (GAP)", "accuracy": 0.707},
    {"source": "Archived PANNs embeddings (this project, full dataset)", "accuracy": 0.837},
    {"source": "Paper — best single classifier (SVM, 3x-aug FDAP)", "accuracy": 0.909},
    {"source": "Paper — best ensemble (3x-aug FDAP)", "accuracy": 0.911},
])

paper_fig = px.bar(
    paper_benchmarks.sort_values("accuracy"),
    x="accuracy",
    y="source",
    orientation="h",
    text="accuracy",
    title="Accuracy: this project's approaches vs. the original paper's reported range",
)
paper_fig.update_traces(texttemplate="%{text:.1%}", textposition="outside")
paper_fig.update_layout(xaxis_title="Accuracy", yaxis_title="", xaxis_tickformat=".0%", xaxis_range=[0, 1])
show_plotly(paper_fig)

## PANNs Embeddings Across Durations

Each `cjk-research-*sec.ipynb` notebook also got a matching **PANNs embeddings** section — a pretrained `Cnn14` model (`panns_inference`, run on GPU) applied to the full 2953-file dataset, cropped/padded to that notebook's own `CLIP_DURATION_S`. The numbers below are the real, already-executed results from each notebook's own 10-fold cross-validation, aggregated here — not recomputed.

In [8]:
pann_duration_results = pd.DataFrame([
    {"clip_duration_s": 2, "model": "svm_rbf", "accuracy": 0.7646, "f1_macro": 0.7614},
    {"clip_duration_s": 2, "model": "random_forest", "accuracy": 0.7667, "f1_macro": 0.7641},
    {"clip_duration_s": 2, "model": "soft_vote", "accuracy": 0.7755, "f1_macro": 0.7733},
    {"clip_duration_s": 5, "model": "svm_rbf", "accuracy": 0.8215, "f1_macro": 0.8208},
    {"clip_duration_s": 5, "model": "random_forest", "accuracy": 0.8232, "f1_macro": 0.8219},
    {"clip_duration_s": 5, "model": "soft_vote", "accuracy": 0.8331, "f1_macro": 0.8324},
    {"clip_duration_s": 7, "model": "svm_rbf", "accuracy": 0.8188, "f1_macro": 0.8175},
    {"clip_duration_s": 7, "model": "random_forest", "accuracy": 0.8280, "f1_macro": 0.8269},
    {"clip_duration_s": 7, "model": "soft_vote", "accuracy": 0.8310, "f1_macro": 0.8302},
    {"clip_duration_s": 11, "model": "svm_rbf", "accuracy": 0.8232, "f1_macro": 0.8216},
    {"clip_duration_s": 11, "model": "random_forest", "accuracy": 0.8229, "f1_macro": 0.8213},
    {"clip_duration_s": 11, "model": "soft_vote", "accuracy": 0.8358, "f1_macro": 0.8350},
])
display(pann_duration_results.pivot(index="clip_duration_s", columns="model", values="accuracy").round(4))

model,random_forest,soft_vote,svm_rbf
clip_duration_s,,,
2,0.7667,0.7755,0.7646
5,0.8232,0.8331,0.8215
7,0.8280,0.8310,0.8188
11,0.8229,0.8358,0.8232


In [9]:
pann_duration_fig = px.line(
    pann_duration_results,
    x="clip_duration_s",
    y="accuracy",
    color="model",
    markers=True,
    title="PANNs embedding accuracy vs. clip duration (full 2953-file dataset, 10-fold CV)",
)
pann_duration_fig.update_layout(xaxis_title="CLIP_DURATION_S (seconds)", yaxis_title="Accuracy", yaxis_tickformat=".0%")
show_plotly(pann_duration_fig)

In [10]:
hand_crafted_long = classifier_results[["clip_duration_s", "model", "accuracy"]].copy()
hand_crafted_long["feature_type"] = "Hand-crafted (8 scalars, 200-file sample)"

pann_long = pann_duration_results[["clip_duration_s", "model", "accuracy"]].copy()
pann_long["feature_type"] = "PANNs embeddings (2048-dim, full dataset)"

combined_duration_df = pd.concat([hand_crafted_long, pann_long], ignore_index=True)

combined_fig = px.line(
    combined_duration_df,
    x="clip_duration_s",
    y="accuracy",
    color="model",
    line_dash="feature_type",
    markers=True,
    title="Hand-crafted features vs. PANNs embeddings: opposite trends as clip duration shrinks",
)
combined_fig.update_layout(xaxis_title="CLIP_DURATION_S (seconds)", yaxis_title="Accuracy", yaxis_tickformat=".0%")
show_plotly(combined_fig)

**The two feature types move in *opposite* directions as duration shrinks — this is the most important finding in this notebook:**

- **PANNs embeddings drop sharply at 2 seconds** (≈77% vs. ≈82–84% at 5/7/11s). The raw dataset's mean clip length is 3.87 seconds, so a 2-second crop genuinely truncates real acoustic content out of many clips — this isn't padding dilution, it's actual information loss, and a deep embedding model is sensitive to it.
- **Hand-crafted features do the opposite: they peak at 2 seconds** (up to 46% accuracy) and drift down as duration grows, because longer windows mean more zero-padded silence diluting the loudness/spectral means for every clip shorter than the window (see the Feature Statistics Sweep above, where every energy feature falls monotonically with duration).
- **Net effect:** there is no single `CLIP_DURATION_S` that is simultaneously optimal for both feature types. Hand-crafted features want short windows to minimize padding; PANNs wants windows long enough to avoid truncating real content (5 seconds or more, based on this data). Whichever feature representation ends up in production should drive the duration choice — not the other way around.

### Why the Difference Exists

Five compounding factors, ranked by how directly the evidence in this project supports them:

1. **Feature representation is the biggest single factor.** The archived PANNs-embeddings run (deep, pretrained CNN14 features) on the *same* originals-only dataset reached 83.7% — close to the paper's own no-augmentation baseline (70.7%) — using nothing but a richer feature representation. This project's hand-crafted 8-scalar means (loudness, noisiness, brightness, timbre) are a much coarser summary than the paper's pooled deep-CNN/CDBN feature maps (GAP/FDAP), and that gap alone explains most of the difference between ~44% and ~84%.
2. **No data augmentation.** The paper's own ablation (Table 3) shows accuracy moving from 70.7% to 91.1% *within the same feature/classifier setup*, purely by adding up to 3x augmented clones per file. This project deliberately works from originals only (team decision), so it never benefits from that lever at all.
3. **Small training sample (for the hand-crafted-feature classifiers).** The classifier cells in every `cjk-research-*sec.ipynb` train on a 200-file stratified subsample (20/class), not the full 2953-file dataset, for speed. Less data means more variance — visible in the ±0.04–0.11 cross-validation std reported above. (The PANNs sweep, by contrast, already uses the full dataset.)
4. **Fixed-length padding dilutes short clips — but only for hand-crafted features.** The feature sweep earlier in this notebook shows `rms`, `zcr`, and every spectral feature falling monotonically as `CLIP_DURATION_S` increases, because more of each short clip becomes zero-padded silence.
5. **Duration interacts with feature type in opposite directions.** The PANNs-across-durations sweep above shows PANNs accuracy actually *drops* at very short windows (2s) due to real content truncation, while hand-crafted features *improve* at short windows due to less padding dilution. The `CLIP_DURATION_S` chosen doesn't just affect accuracy in isolation — it interacts with whichever feature representation is eventually chosen for production, in opposite directions.

### What Would Tighten the Model

Concrete, ordered by expected impact based on the evidence above:

1. **Swap hand-crafted features for deep embeddings.** The single largest lever measured in this project: PANNs embeddings hit 82–84% at 5s/7s/11s vs. this project's hand-crafted-feature range of 41.5–46.0% — nearly double, using 2048-dim pretrained CNN embeddings instead of 8 hand-crafted scalars. This is the highest-value next step.
2. **If moving to deep embeddings, avoid very short capture windows.** The duration sweep above shows PANNs specifically loses about 6 points of accuracy at a 2-second window (≈77%) versus 5 seconds or longer (≈82–84%) — the latency argument for a short capture window (from the earlier sample-rate/duration API discussion) applies to hand-crafted features, not to deep embeddings, where 5+ seconds performed best.
3. **Add data augmentation.** The paper's own ablation shows this is its biggest lever too: 70.7% → 91.1% within the *same* feature/classifier setup, just by adding 3x augmented clones. This project currently excludes augmentation entirely (team decision, originals only).
4. **Train the hand-crafted-feature classifiers on the full 2953-file dataset, not a 200-file subsample.** The "Classifier Sanity Checks" sections all sample 20 files/class for speed — more data should reduce the ±0.04–0.11 cross-validation std observed and likely lift accuracy on its own. (Not needed for the PANNs sweep, which already runs on the full dataset.)
5. **Use group-aware cross-validation.** 149 segment groups (398 files) are pieces of the same recording. `StratifiedKFold` doesn't know this — pieces of the same clip can land in both train and test folds. Switching to `StratifiedGroupKFold` on `group` would give a more trustworthy accuracy estimate (could move the number in either direction) for both feature types.
6. **Replace center-crop/pad with loudest-window extraction — for hand-crafted features specifically.** The feature sweep above shows every energy-based feature (`rms`, `zcr`, spectral stats) drops monotonically as `CLIP_DURATION_S` grows, because short clips get diluted with zero-padded silence. Extracting the loudest N-second window instead of a centered one would stop that self-inflicted signal loss — note this fix doesn't apply to the PANNs finding, which is about content *truncation* at short windows, a different problem.
7. **Tune hyperparameters.** Every model here (`SVC`, `RandomForestClassifier`, `VotingClassifier`) uses near-default parameters. A grid/random search over `SVC`'s `C`/`gamma` and the forest's depth/`n_estimators` costs little and wasn't attempted anywhere in this project yet.

## Dataset Context (duration-independent, cited from `cjk-research-2sec.ipynb`)

These facts don't change with `CLIP_DURATION_S` since they describe the raw dataset before any cropping/padding — see `cjk-research-2sec.ipynb` for the full analysis and interactive charts:

- **2953 files across 10 classes**, originals only (team decision 08.09.2026), no exact duplicates remaining.
- **149 segment groups / 398 files** are pieces of the same longer recording cut into pieces — must stay grouped in any train/test split, not split randomly row-by-row.
- **Native sample rate**: 98.7% of files are 44100 Hz; a 1.3% minority sits at 8000-32000 Hz. `SAMPLE_RATE = 44100` (used in every `cjk-research-*sec.ipynb` variant, including this sweep) maximizes fidelity, only upsampling that 1.3% minority rather than downsampling the dominant majority.
- **Raw duration**: mean 3.87s, p95 6.82s across the uncropped files — the basis for the historical `CLIP_DURATION_S = 7.0` recommendation, now empirically tested here against the 2/5/11s alternatives above.